In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing


housing=fetch_california_housing()
print(housing)


{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

In [2]:
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [5]:
# Preparing the dataset

data=pd.DataFrame(housing.data,columns=housing.feature_names)
data.sample(10)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
17386,1.7941,22.0,4.216146,0.937500,1509.0,3.929688,34.97,-120.44
14096,2.0500,11.0,3.571111,1.062222,1384.0,3.075556,32.75,-117.11
12883,3.0417,35.0,4.798507,0.873134,331.0,2.470149,38.67,-121.34
10590,5.1299,16.0,5.776413,0.990172,2529.0,3.106880,33.69,-117.78
3361,2.2661,16.0,5.783051,1.130508,1873.0,3.174576,40.37,-120.58
281,6.7527,23.0,7.064024,1.024390,955.0,2.911585,37.80,-122.18
11385,7.3719,22.0,7.104592,1.125000,1086.0,2.770408,33.66,-117.95
15755,3.3500,52.0,4.783002,1.132007,1275.0,2.305606,37.77,-122.45
1588,8.0030,18.0,7.812339,1.069409,3280.0,2.810626,37.82,-121.98
2113,1.4107,29.0,4.247444,1.071575,1887.0,3.858896,36.76,-119.75


In [7]:
data['price']=housing.target
data.sample(10)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
12890,3.6786,27.0,5.929078,1.060284,757.0,2.684397,38.64,-121.37,1.597
3938,5.6345,35.0,5.470085,0.942308,1428.0,3.051282,34.22,-118.58,2.282
11092,3.7604,15.0,5.804143,1.009416,1268.0,2.387947,33.81,-117.87,2.801
4102,5.5683,19.0,5.287500,1.077083,2021.0,2.105208,34.14,-118.39,3.092
15261,6.3712,9.0,6.762044,1.108029,1724.0,2.516788,33.02,-117.26,3.698
737,3.1734,38.0,6.060241,1.045181,880.0,2.650602,37.67,-122.13,1.816
14751,4.6364,17.0,6.169863,1.035616,1776.0,4.865753,32.57,-117.06,1.411
1566,15.0001,2.0,22.222222,2.222222,25.0,2.777778,37.74,-121.96,3.500
8539,4.0727,18.0,3.957845,1.079625,2276.0,2.665105,33.90,-118.36,2.140
3153,3.2405,16.0,6.113772,1.215569,2033.0,3.043413,35.12,-118.46,0.855


In [8]:
from urllib.parse import urlparse
X=data.drop(columns=['price'])
y=data['price']


In [9]:
xtrain,xtest,ytrain,ytest=train_test_split(X,y,random_state=42,test_size=0.2)
from mlflow.models import infer_signature
signature=infer_signature(xtrain,ytrain)

param_grid={
    'n_estimators':[100,200],
    'max_depth':[5,10,None],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2]
}

In [11]:
# Hyper parameter tuning using gridsearchCV

def hyperParameterTuner(xtrain,ytrain,param_grid):
    rf=RandomForestRegressor()
    gridsearch=GridSearchCV(estimator=rf,param_grid=param_grid,cv=5,n_jobs=1,verbose=2,scoring="neg_mean_squared_error")

    gridserach.fit(xtrain,ytrain)
    return gridsearch

In [ ]:
with mlflow.start_run():
    grid_search=hyperParameterTuner(xtrain,ytrain,param_grid)
    # get the best model
    best_model=grid_search.best_estimator_
    # evaluate the best model
    ypred=best_model.predict(xtest)
    mse=mean_squared_error(ytest,ypred)
    # log best parameters and metrics
    mlflow.log_param("best_n_estimators",grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth",grid_search.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split",grid_search.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf",grid_search.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse",mse)

    # Tracking url